In [1]:
from pathlib import Path
import pandas as pd
import re
import ast
import re
import numpy as np

In [2]:
dw_df = pd.read_csv('../data/minimal_dose_cluster_dataset.csv')
dw_df.head()

,id,D10,D15,D2,D20,D25,D30,D35,D40,D45,...,symptoms_skin,symptoms_sleep,symptoms_sob,symptoms_swallow,symptoms_taste,symptoms_teeth,symptoms_voice,symptoms_vomit,symptoms_walking,symptoms_work
0,2,"[51.625, 26.890625, 56.1875, 53.625, 53.75, 57...","[49.78125, 26.515625, 54.6875, 52.21875, 52.96...","[54.53125, 28.1875, 59.84375, 57.40625, 55.937...","[47.09375, 25.875, 53.09375, 50.875, 52.4375, ...","[44.4375, 25.453125, 51.375, 49.75, 52.03125, ...","[42.53125, 25.078125, 49.71875, 48.6875, 51.65...","[41.03125, 24.703125, 48.03125, 47.75, 51.3125...","[39.78125, 24.375, 46.46875, 46.625, 51.0, 51....","[38.4375, 24.125, 44.875, 45.6875, 50.5625, 51...",...,"[0.0, 0.0, 2.0, 1.0, 1.0, 2.0, 1.0, 5.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 1.0, 1.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","[3.0, 1.0, 1.0, 0.0, 5.0, 5.0, 2.0, 7.0, 4.0, ...","[3.0, 1.0, 1.0, 4.0, 3.0, 4.0, 3.0, 7.0, 5.0, ...","[0.0, 0.0, 1.0, 0.0, 3.0, 3.0, 2.0, 6.0, 4.0, ...","[0.0, 0.0, 0.0, 0.0, 3.0, 4.0, 3.0, 3.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 3.0, 0.0, ...","[0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, ...","[0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 4.0, 3.0, ..."
1,7,"[37.625, 37.15625, 66.3125, 60.3125, 49.4375, ...","[36.5625, 36.75, 64.0625, 59.96875, 48.375, 49...","[39.96875, 38.0625, 71.8125, 60.90625, 52.1562...","[35.25, 36.34375, 62.71875, 59.71875, 47.34375...","[34.0625, 35.90625, 61.53125, 59.40625, 46.562...","[33.125, 35.5, 60.34375, 59.1875, 45.6875, 43....","[32.28125, 35.03125, 59.03125, 58.90625, 44.75...","[31.609375, 34.625, 57.71875, 58.625, 43.96875...","[30.84375, 34.1875, 56.40625, 58.3125, 43.1875...",...,"[0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 5.0, 5.0, 0.0, ...","[0.0, 2.0, 5.0, 2.0, 1.0, 2.0, 2.0, 2.0, 1.0, ...","[3.0, 1.0, 2.0, 1.0, 2.0, 1.0, 3.0, 0.0, 1.0, ...","[0.0, 0.0, 1.0, 4.0, 6.0, 5.0, 8.0, 1.0, 2.0, ...","[5.0, 3.0, 3.0, 5.0, 8.0, 6.0, 7.0, 7.0, 6.0, ...","[2.0, 0.0, 0.0, 3.0, 3.0, 0.0, 0.0, 1.0, 1.0, ...","[0.0, 1.0, 1.0, 1.0, 2.0, 3.0, 4.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 1.0, 5.0, 0.0, 2.0, 1.0, 0.0, ...","[0.0, 0.0, 0.0, 2.0, 1.0, 0.0, 1.0, 0.0, 1.0, ...","[0.0, 0.0, 0.0, 1.0, 2.0, 1.0, 2.0, 0.0, 2.0, ..."
2,8,"[32.15625, 32.0625, 65.375, 56.6875, 22.5, 31....","[31.703125, 31.109375, 63.34375, 55.75, 21.312...","[33.4375, 34.09375, 68.75, 58.28125, 33.6875, ...","[31.203125, 30.25, 61.34375, 54.5625, 17.84375...","[30.609375, 29.40625, 60.03125, 52.375, 14.367...","[29.9375, 28.390625, 59.5, 50.5625, 13.140625,...","[28.96875, 27.453125, 58.96875, 49.46875, 11.7...","[27.1875, 26.46875, 58.53125, 48.59375, 11.203...","[24.875, 25.453125, 58.125, 48.0, 10.9609375, ...",...,"[0.0, 0.0, 1.0, 1.0, 1.0, 2.0, 3.0, 3.0, 1.0, ...","[2.0, 2.0, 2.0, 1.0, 2.0, 1.0, 4.0, 5.0, 3.0, ...","[1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, ...","[1.0, 6.0, 1.0, 1.0, 3.0, 2.0, 3.0, 5.0, 4.0, ...","[1.0, 2.0, 2.0, 3.0, 2.0, 3.0, 2.0, 2.0, 4.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 2.0, 0.0, 0.0, 2.0, 1.0, 1.0, 4.0, 0.0, ...","[0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 3.0, 5.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, ...","[1.0, 1.0, 1.0, 0.0, 2.0, 2.0, 0.0, 2.0, 3.0, ..."
3,9,"[51.4375, 27.515625, 51.4375, 51.4375, 18.6875...","[48.59375, 26.671875, 48.59375, 48.59375, 15.0...","[56.65625, 29.5625, 56.65625, 56.65625, 27.312...","[46.03125, 25.9375, 46.03125, 46.03125, 13.0, ...","[43.53125, 25.296875, 43.53125, 43.53125, 11.3...","[41.28125, 24.671875, 41.28125, 41.28125, 10.3...","[39.15625, 24.046875, 39.15625, 39.15625, 9.82...","[37.0, 23.359375, 37.0, 37.0, 9.4609375, 11.28...","[35.125, 22.625, 35.125, 35.125, 9.109375, 9.3...",...,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 2.0, 0.0, 0.0, ...","[0.0, 0.0, 1.0, 1.0, 1.0, 2.0, 5.0, 5.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","[0.0, 2.0, 1.0, 2.0, 2.0, 2.0, 3.0, 3.0, 3.0, ...","[0.0, 2.0, 3.0, 4.0, 6.0, 5.0, 5.0, 9.0, 4.0, ...","[0.0, 0.0, 0.0, 1.0, 1.0, 3.0, 2.0, 2.0, 2.0, ...","[0.0, 2.0, 1.0, 1.0, 1.0, 1.0,

In [3]:
merged_df = pd.read_csv('./old_dose_20260421.csv')
merged_df.head()

,id,volume,mean_dose,min_dose,max_dose,V5,V10,V15,V20,V25,...,symptoms_skin,symptoms_sleep,symptoms_sob,symptoms_swallow,symptoms_taste,symptoms_teeth,symptoms_voice,symptoms_vomit,symptoms_walking,symptoms_work
0,24,"[14724.42626953125, 14765.625, 6632.9956054687...","[22.765716284275324, 33.629034598214275, 56.96...","[1.0, 2114.0, 4179.0, 3625.0, 3459.0, 2969.0, ...","[4362.0, 3864.0, 6126.0, 7216.0, 5552.0, 5877....","[80.0625, 100.0, 100.0, 100.0, 100.0, 100.0, 1...","[70.3125, 100.0, 100.0, 100.0, 100.0, 100.0, 1...","[64.75, 100.0, 100.0, 100.0, 100.0, 100.0, 100...","[61.5, 100.0, 100.0, 100.0, 100.0, 100.0, 100....","[58.03125, 98.5, 100.0, 100.0, 100.0, 100.0, 1...",...,"[0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 5.0, 3.0, 0.0, ...","[0.0, 2.0, 5.0, 2.0, 1.0, 2.0, 2.0, 1.0, 1.0, ...","[3.0, 1.0, 2.0, 1.0, 2.0, 0.0, 3.0, 0.0, 1.0, ...","[0.0, 0.0, 1.0, 4.0, 6.0, 3.0, 8.0, 1.0, 2.0, ...","[5.0, 3.0, 3.0, 5.0, 8.0, 5.0, 7.0, 6.0, 6.0, ...","[2.0, 0.0, 0.0, 3.0, 3.0, 0.0, 0.0, 0.0, 1.0, ...","[0.0, 1.0, 1.0, 1.0, 2.0, 2.0, 4.0, 0.0, 1.0, ...","[0.0, 0.0, 0.0, 1.0, 5.0, 1.0, 2.0, 4.0, 0.0, ...","[0.0, 0.0, 0.0, 2.0, 1.0, 0.0, 1.0, 0.0, 1.0, ...","[0.0, 0.0, 0.0, 1.0, 2.0, 0.0, 2.0, 0.0, 2.0, ..."
1,26,"[10428.309631347656, 17159.777069091797, 4182....","[19.083991985203443, 21.81411015361559, 48.068...","[352.0, 572.0, 909.0, 3685.0, 599.0, 609.0, 38...","[3401.0, 3709.0, 5978.0, 7046.0, 3621.0, 4159....","[95.4375, 100.0, 100.0, 100.0, 100.0, 100.0, 1...","[61.53125, 78.6875, 99.9375, 100.0, 48.46875, ...","[55.34375, 75.625, 99.75, 100.0, 24.296875, 33...","[49.1875, 70.625, 99.0625, 100.0, 18.40625, 22...","[44.75, 47.5, 98.75, 100.0, 5.3359375, 16.25, ...",...,"[0.0, 0.0, 0.0, 1.0, 2.0, 2.0, 1.0, 2.0, 1.0, ...","[2.0, 2.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 3.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 6.0, 0.0, 0.0, 0.0, 0.0, 2.0, 2.0, 4.0, ...","[1.0, 2.0, 1.0, 1.0, 1.0, 3.0, 3.0, 2.0, 4.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, ...","[0.0, 2.0, 1.0, 1.0, 1.0, 0.0, 0.0, 2.0, 0.0, ...","[0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 2.0, 0.0, ...","[0.0, 0.0, 2.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","[1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 3.0, ..."
2,28,"[10272.979736328125, 21972.10121154785, 10272....","[33.69805989786887, 20.803234820775426, 33.698...","[1075.0, 754.0, 1075.0, 1075.0, 532.0, 484.0, ...","[5966.0, 3283.0, 5966.0, 5966.0, 3305.0, 5288....","[100.0, 100.0, 100.0, 100.0, 100.0, 98.5, 100....","[100.0, 95.0, 100.0, 100.0, 33.375, 42.9375, 6...","[96.5, 79.875, 96.5, 96.5, 15.3671875, 34.4062...","[86.1875, 62.75, 86.1875, 86.1875, 8.390625, 2...","[73.4375, 27.28125, 73.4375, 73.4375, 3.335937...",...,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 2.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 2.0, 5.0, 5.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","[0.0, 2.0, 1.0, 1.0, 2.0, 2.0, 3.0, 3.0, 3.0, ...","[0.0, 2.0, 1.0, 3.0, 6.0, 5.0, 5.0, 9.0, 4.0, ...","[0.0, 0.0, 0.0, 1.0, 1.0, 3.0, 2.0, 2.0, 2.0, ...","[0.0, 2.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 2.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 2.0, 1.0, ...","[0.0, 0.0, 1.0, 0.0, 3.0, 3.0, 3.0, 2.0, 1.0, ..."
3,34,"[16405.33447265625, 21472.7783203125, 13253.63...","[27.220497237569013, 29.323211818879553, 40.88...","[307.0, 2155.0, 594.0, 1120.0, 3675.0, 3087.0,...","[5181.0, 4611.0, 7108.0, 5900.0, 6108.0, 6322....","[91.5625, 100.0, 100.0, 100.0, 100.0, 100.0, 1...","[75.1875, 100.0, 85.8125, 100.0, 100.0, 100.0,...","[68.9375, 100.0, 79.9375, 94.9375, 100.0, 100....","[64.6875, 100.0, 74.6875, 88.6875, 100.0, 100....","[60.84375, 81.8125, 69.1875, 83.9375, 100.0, 1...",...,"[0.0, 0.0, 0.0, 1.0, 5.0, 3.0, 4.0, 2.0, 0.0, ...","[3.0, 0.0, 0.0, 2.0, 1.0, 0.0, 3.0, 3.0, 0.0, ...","[0.0, 3.0, 1.0, 3.0, 2.0, 2.0, 3.0, 4.0, 0.0, ...","[0.0, 0.0, 0.0, 3.0, 3.0, 4.0, 6.0, 5.0, 1.0, ...","[0.0, 0.0, 0.0, 2.0, 6.0, 7.0, 7.0, 0.0, 2.0, ...","[0.0, 0.0, 0.0, 2.0, 2.0, 3.0, 2.0, 6.0, 1.0, ...","[2.0, 0

In [4]:
def patient_ids(df: pd.DataFrame) -> set:
    """Normalize patient id sets whether id is a column or the index."""
    if "id" in df.columns:
        s = df["id"]
    else:
        s = df.index
    return set(pd.Series(s).dropna().astype(int))


ids_dw = patient_ids(dw_df)
ids_merged = patient_ids(merged_df)

# In DW but dropped from merged_df (inner merge / filtering upstream)
ids_in_dw_missing_from_merged = sorted(ids_dw - ids_merged)
# In merged_df but not listed as patient rows in DW export (unexpected)
ids_in_merged_not_in_dw = sorted(ids_merged - ids_dw)

print(f"DW rows (unique ids): {len(ids_dw)}, merged_df rows (unique ids): {len(ids_merged)}")
print(f"\nIDs in DW [CSV] but NOT in merged_df ({len(ids_in_dw_missing_from_merged)}):")
print(ids_in_dw_missing_from_merged)
print(f"\nIDs in merged_df but NOT in DW CSV ({len(ids_in_merged_not_in_dw)}):")
print(ids_in_merged_not_in_dw)

DW rows (unique ids): 349, merged_df rows (unique ids): 337

IDs in DW [CSV] but NOT in merged_df (229):
[2, 7, 8, 9, 10, 12, 13, 15, 16, 18, 19, 21, 22, 23, 25, 29, 31, 32, 35, 37, 39, 40, 41, 43, 44, 46, 48, 49, 50, 51, 52, 53, 56, 57, 58, 61, 62, 67, 68, 69, 71, 77, 79, 81, 86, 92, 96, 97, 102, 107, 111, 114, 116, 117, 120, 123, 126, 133, 134, 135, 136, 138, 139, 140, 143, 145, 146, 150, 163, 166, 168, 176, 177, 184, 185, 190, 194, 198, 200, 211, 214, 216, 218, 221, 224, 226, 230, 231, 232, 233, 237, 240, 241, 246, 249, 251, 254, 256, 258, 259, 260, 262, 263, 268, 269, 270, 273, 280, 284, 285, 287, 290, 291, 294, 303, 304, 311, 313, 315, 316, 320, 323, 324, 325, 327, 329, 333, 335, 344, 345, 346, 349, 352, 353, 361, 362, 363, 365, 366, 374, 375, 376, 379, 383, 386, 388, 389, 390, 395, 397, 399, 401, 402, 403, 406, 407, 410, 412, 414, 419, 421, 424, 425, 428, 434, 447, 450, 454, 466, 467, 471, 474, 475, 486, 487, 489, 495, 496, 498, 500, 507, 509, 514, 515, 517, 518, 529, 534, 535, 5

In [9]:
cols_dw = set(dw_df.columns)
cols_merged = set(merged_df.columns)

# Columns present in DW export but absent from merged_df schema
cols_in_dw_missing_from_merged = sorted(cols_dw - cols_merged)
cols_only_in_merged = sorted(cols_merged - cols_dw)

print(f"DW column count: {len(cols_dw)}, merged_df column count: {len(cols_merged)}")
print(f"\nColumns in DW [CSV] but NOT in merged_df ({len(cols_in_dw_missing_from_merged)}):")
print(cols_in_dw_missing_from_merged)
print(f"\nColumns only in merged_df (not in DW CSV) ({len(cols_only_in_merged)}):")
print(cols_only_in_merged)

DW column count: 152, merged_df column count: 119

Columns in DW [CSV] but NOT in merged_df (35):
['IMPT', 'IMRT', 'Local_control', 'Regional_control', 'Technique', 'VMAT', 'age>median', 'age_65', 'baseline_bmi', 'baseline_height', 'baseline_weight', 'bmi_change', 'chemotherapy', 'concurrent_prior_to_enrollment', 'end_of_treatment_bmi', 'end_of_treatment_weight', 'ic_prior_to_enrollment', 'level_0', 'longterm_bmi_change', 'n2', 'n_severe', 'os', 'performance_1', 'performance_2', 'performance_high', 'performance_score', 'previously_treated', 'status_at_enrollememt', 'sx_prior_to_enrollment', 'sxprimary', 't3', 't_severe', 'weight_loss_5kg', 'wk6_bmi', 'wk6_weight']

Columns only in merged_df (not in DW CSV) (2):
['baseline_mbs_digest', 'old']


In [10]:
def to_array_len(x):
    """Return length for list-like content stored as list/tuple/ndarray or stringified list."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (list, tuple, np.ndarray)):
        return len(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return np.nan
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, np.ndarray)):
                return len(parsed)
        except (ValueError, SyntaxError):
            return np.nan
    return np.nan

In [12]:
# 1) Align dw_df columns to merged_df (same names + same order)
dw_extra_cols = [c for c in dw_df.columns if c not in merged_df.columns]
dw_missing_cols = [c for c in merged_df.columns if c not in dw_df.columns]

print(f"DW extra columns (will be dropped): {len(dw_extra_cols)}")
print(dw_extra_cols[:20], "..." if len(dw_extra_cols) > 20 else "")
print(f"Columns in merged_df but missing from DW (will become NaN after reindex): {len(dw_missing_cols)}")
print(dw_missing_cols[:20], "..." if len(dw_missing_cols) > 20 else "")

dw_df = dw_df.reindex(columns=merged_df.columns)
print("\nAfter alignment:")
print("dw_df column count:", len(dw_df.columns))
print("merged_df column count:", len(merged_df.columns))
print("Columns exactly same:", list(dw_df.columns) == list(merged_df.columns))

DW extra columns (will be dropped): 0
[] 
Columns in merged_df but missing from DW (will become NaN after reindex): 0
[] 

After alignment:
dw_df column count: 119
merged_df column count: 119
Columns exactly same: True


In [14]:
# 2) Compare array lengths between dw_df and merged_df on shared ids
if "id" not in dw_df.columns or "id" not in merged_df.columns:
    raise ValueError("Both dw_df and merged_df must contain an 'id' column for row-wise comparison.")

# Candidate columns:
# - explicit names: volume / mean dose / mean_dose
# - dose-volume family: V5, V10, ... ; D5, D10, ...
# - symptom family: symptoms_* or symtoms_* (typo tolerant)
explicit_candidates = ["volume", "mean dose", "mean_dose"]
pattern_candidates = [
    c for c in dw_df.columns
    if re.match(r"^(V\d+|D\d+)$", str(c)) or re.match(r"^sym(?:p|t)oms?_", str(c))
]

candidate_cols = sorted(set(explicit_candidates + pattern_candidates))
array_cols = [c for c in candidate_cols if c in dw_df.columns and c in merged_df.columns]

print(f"Array-like columns to check: {len(array_cols)}")
print(array_cols)

# Restrict to shared ids so both sides are comparable
shared_ids = sorted(set(dw_df["id"].dropna().astype(int)) & set(merged_df["id"].dropna().astype(int)))
print(f"\nShared ids for comparison: {len(shared_ids)}")

dw_idx = dw_df.copy()
mg_idx = merged_df.copy()
dw_idx["id"] = dw_idx["id"].astype(int)
mg_idx["id"] = mg_idx["id"].astype(int)
dw_idx = dw_idx.set_index("id")
mg_idx = mg_idx.set_index("id")

mismatches = []
for pid in shared_ids:
    for col in array_cols:
        len_dw = to_array_len(dw_idx.at[pid, col]) if pid in dw_idx.index else np.nan
        len_mg = to_array_len(mg_idx.at[pid, col]) if pid in mg_idx.index else np.nan

        # If one side parses to a list length and the other does not, or lengths differ, mark mismatch
        if (pd.isna(len_dw) and not pd.isna(len_mg)) or (not pd.isna(len_dw) and pd.isna(len_mg)):
            mismatches.append((pid, col, len_dw, len_mg, "unparsed_or_non_array"))
        elif (not pd.isna(len_dw)) and (not pd.isna(len_mg)) and (int(len_dw) != int(len_mg)):
            mismatches.append((pid, col, int(len_dw), int(len_mg), "length_mismatch"))

mismatch_df = pd.DataFrame(
    mismatches,
    columns=["id", "column", "dw_len", "merged_len", "issue"]
)

if mismatch_df.empty:
    print("\nAll checked array columns have matching array lengths between dw_df and merged_df.")
else:
    print(f"\nFound mismatches: {len(mismatch_df)}")
    display(mismatch_df.sort_values(["column", "id"]).head(200))

# Optional quick summary by column
if not mismatch_df.empty:
    display(mismatch_df.groupby(["column", "issue"]).size().reset_index(name="n").sort_values("n", ascending=False))

Array-like columns to check: 41
['D10', 'D15', 'D2', 'D20', 'D25', 'D30', 'D35', 'D40', 'D45', 'D5', 'D50', 'D55', 'D60', 'D65', 'D70', 'D75', 'D80', 'D85', 'D90', 'D95', 'D97', 'D98', 'D99', 'V10', 'V15', 'V20', 'V25', 'V30', 'V35', 'V40', 'V45', 'V5', 'V50', 'V55', 'V60', 'V65', 'V70', 'V75', 'V80', 'mean_dose', 'volume']

Shared ids for comparison: 120

All checked array columns have matching array lengths between dw_df and merged_df.


In [ ]:
# 3) Replace dw_df id with Stiefel_ID (same method as PreProcessing_Tutorial),
#    then strip 'STIEFEL_' and save as a new CSV.
_candidates = (
    Path.cwd() / "Cohort_SMART2_530pts_(486pts) - STIEFEL_ID_Anonymized_20260421.xlsx",
    Path.cwd().parent / "Cohort_SMART2_530pts_(486pts) - STIEFEL_ID_Anonymized_20260421.xlsx",
)
_COHORT_STIEFEL_XLSX = next((p for p in _candidates if p.is_file()), _candidates[-1])

_stiefel_map = (
    pd.read_excel(_COHORT_STIEFEL_XLSX, sheet_name=0, usecols=["id", "Stiefel_ID"])
    .drop_duplicates(subset=["id"])
)
_stiefel_map["_id_key"] = _stiefel_map["id"].astype(float)

dw_df_stiefel = dw_df.copy()
if "id" not in dw_df_stiefel.columns:
    raise ValueError("dw_df has no 'id' column.")

_orig_id = dw_df_stiefel["id"].copy()
dw_df_stiefel["_id_key"] = pd.to_numeric(dw_df_stiefel["id"], errors="coerce").astype(float)
dw_df_stiefel = dw_df_stiefel.merge(
    _stiefel_map[["_id_key", "Stiefel_ID"]],
    on="_id_key",
    how="left",
    validate="m:1",
)

_missing = dw_df_stiefel["Stiefel_ID"].isna()
if _missing.any():
    _bad = _orig_id[_missing].drop_duplicates().tolist()
    print(
        f"Warning: {_missing.sum()} row(s) had no Stiefel_ID match; "
        f"unique id values: {_bad[:40]}"
        + (" ..." if len(_bad) > 40 else "")
    )

# Same conversion style as tutorial: STIEFEL_XXXX -> XXXX (int)
dw_df_stiefel["id"] = dw_df_stiefel["Stiefel_ID"].fillna(_orig_id)
dw_df_stiefel["id"] = (
    dw_df_stiefel["id"]
    .astype(str)
    .str.replace("STIEFEL_", "", regex=False)
    .astype(int)
)

dw_df_stiefel = dw_df_stiefel.drop(columns=["_id_key", "Stiefel_ID"])

out_csv = Path("./dw_df_stiefel_id_20260421.csv")
dw_df_stiefel.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")
dw_df_stiefel.head(2)

Saved: C:\Users\szhao69\Documents\GitHub\QubbedDataAnalysis\python\dw_df_stiefel_id_20260430.csv


,id,volume,mean_dose,min_dose,max_dose,V5,V10,V15,V20,V25,...,symptoms_skin,symptoms_sleep,symptoms_sob,symptoms_swallow,symptoms_taste,symptoms_teeth,symptoms_voice,symptoms_vomit,symptoms_walking,symptoms_work
0,8,"[14279.47998046875, 12866.363525390625, 7139.7...","[35.72055395268319, 23.816881203970524, 36.247...","[2.97, 17.68, 3.41, 13.08, 40.7, 25.79, 28.34,...","[58.27, 30.08, 62.72, 60.34, 57.47, 68.42, 59....","[96.375, 100.0, 95.6875, 100.0, 100.0, 100.0, ...","[91.5625, 100.0, 80.0, 100.0, 100.0, 100.0, 10...","[88.25, 100.0, 75.5, 99.0, 100.0, 100.0, 100.0...","[85.6875, 93.875, 72.5625, 98.375, 100.0, 100....","[82.9375, 31.1875, 70.3125, 96.25, 100.0, 100....",...,"[0.0, 0.0, 2.0, 1.0, 1.0, 2.0, 1.0, 5.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 1.0, 1.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","[3.0, 1.0, 1.0, 0.0, 5.0, 5.0, 2.0, 7.0, 4.0, ...","[3.0, 1.0, 1.0, 4.0, 3.0, 4.0, 3.0, 7.0, 5.0, ...","[0.0, 0.0, 1.0, 0.0, 3.0, 3.0, 2.0, 6.0, 4.0, ...","[0.0, 0.0, 0.0, 0.0, 3.0, 4.0, 3.0, 3.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 3.0, 0.0, ...","[0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, ...","[0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 4.0, 3.0, ..."
1,24,"[14724.42626953125, 14765.625, 6645.3552246093...","[22.765716284275324, 33.629034598214275, 55.53...","[1.0, 21.14, 36.25, 41.79, 34.59, 29.69, 36.68...","[43.62, 38.64, 72.16, 61.26, 55.52, 58.77, 67....","[80.0625, 100.0, 100.0, 100.0, 100.0, 100.0, 1...","[70.3125, 100.0, 100.0, 100.0, 100.0, 100.0, 1...","[64.75, 100.0, 100.0, 100.0, 100.0, 100.0, 100...","[61.5, 100.0, 100.0, 100.0, 100.0, 100.0, 100....","[58.03125, 98.5, 100.0, 100.0, 100.0, 100.0, 1...",...,"[0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 5.0, 5.0, 0.0, ...","[0.0, 2.0, 5.0, 2.0, 1.0, 2.0, 2.0, 2.0, 1.0, ...","[3.0, 1.0, 2.0, 1.0, 2.0, 1.0, 3.0, 0.0, 1.0, ...","[0.0, 0.0, 1.0, 4.0, 6.0, 5.0, 8.0, 1.0, 2.0, ...","[5.0, 3.0, 3.0, 5.0, 8.0, 6.0, 7.0, 7.0, 6.0, ...","[2.0, 0.0, 0.0, 3.0, 3.0, 0.0, 0.0, 1.0, 1.0, ...","[0.0, 1.0, 1.0, 1.0, 2.0, 3.0, 4.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 1.0, 5.0, 0.0, 2.0, 1.0, 0.0, ...","[0.0, 0.0, 0.0, 2.0, 1.0, 0.0, 1.0, 0.0, 1.0, ...","[0.0, 0.0, 0.0, 1.0, 2.0, 1.0, 2.0, 0.0, 2.0, ..."
